# LintGate Mutation Sweep

**Auto-generated** for `main` branch. Fully self-contained.

- Source files to profile: **400**
- Already cached locally: **2679**
- Estimated new profiles: **~0**

**How to use:** Runtime > Run all. Wait. Download the zip at the end.
Then: `cd . && unzip ~/Downloads/mutation_results.zip`


## Step 1: Clone & install


In [ ]:
import importlib, os, shutil, subprocess, sys

REPO_URL = "https://github.com/rohanvinaik/LintGate.git"
BRANCH = "main"
PROJECT_DIR = "/content/lintgate"

# Uncomment if repo is private:
# GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

clone_url = REPO_URL
try:
    GITHUB_TOKEN
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    print('Using authenticated clone')
except NameError:
    print('Using public clone')

result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, clone_url, PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f'Branch {BRANCH} failed, trying default branch...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'Clone failed: {result.stderr}')
        raise RuntimeError('Clone failed')
print(f'Cloned to {PROJECT_DIR}')

# Install deps
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'hatchling', 'pyyaml', 'packaging'],
    check=True
)
# Try editable install but don't rely on it
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT_DIR],
    capture_output=True, text=True
)

# Always ensure PROJECT_DIR is on sys.path (belt and suspenders)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Purge any stale module cache entries, then invalidate import caches
for m in list(sys.modules):
    if m.startswith('lintgate') or m.startswith('mcp_tools'):
        del sys.modules[m]
importlib.invalidate_caches()

from lintgate.specification.mutation_engine import MutationCategory
print(f'Import OK. Categories: {[c.value for c in MutationCategory]}')
print('Ready for Step 2!')


## Step 2: Run sweep


In [ ]:
import ast, json, os, sys, time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

WORKERS = 4
BUDGET_MS = 500.0
SKIP_CACHED = False  # Fresh clone — profile everything

def collect_python_files(root):
    files = []
    for subdir in ('lintgate', 'mcp_tools'):
        base = os.path.join(root, subdir)
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            for fn in sorted(filenames):
                if fn.endswith('.py') and not fn.startswith('test_') and not fn.endswith('_test.py'):
                    files.append(os.path.relpath(os.path.join(dirpath, fn), root))
    return files

def profile_single_file(args):
    project_root, rel_path, budget_ms, skip_cached = args
    import ast, os, sys, time
    sys.path.insert(0, project_root)
    from lintgate.specification.mutation_engine import run_function_sampling
    from lintgate.keys import canonical_function_key
    from lintgate.specification.mutation_filter import filter_categories
    from mcp_tools._mutation_impl import (
        MutationContext, detect_purity_map, discover_test_files,
        get_cache_dir, load_test_callables, lookup_purity,
        parse_file, save_cached_state, walk_functions,
    )
    full_path = os.path.join(project_root, rel_path)
    cache_dir = get_cache_dir(project_root)
    start = time.monotonic()
    result = {'file': rel_path, 'profiled': 0, 'cached': 0, 'trivial': 0,
              'killed': 0, 'survived': 0, 'errors': 0, 'discovery': {}}
    tree = parse_file(full_path)
    if tree is None:
        result['errors'] = 1
        return result
    functions = walk_functions(tree)
    if not functions:
        return result
    test_files = discover_test_files(project_root, full_path)
    purity_map = detect_purity_map(full_path)
    ctx = MutationContext(
        full_path=full_path, rel_path=rel_path, cache_dir=cache_dir,
        purity_map=purity_map, test_files=test_files, project_root=project_root,
    )
    for qualname, node in functions:
        func_key = canonical_function_key(rel_path, qualname)
        if skip_cached:
            safe_key = func_key.replace('::', '__').replace('/', '_')
            if (cache_dir / f'{safe_key}.json').exists():
                result['cached'] += 1
                continue
        body = getattr(node, 'body', [])
        if len(body) <= 1:
            stmt = body[0] if body else None
            if isinstance(stmt, (ast.Return, ast.Expr)):
                result['trivial'] += 1
                continue
        is_pure = lookup_purity(purity_map, qualname)
        cats = filter_categories(node, is_pure=is_pure)
        bare_name = qualname.split('.')[-1]
        tests, diag = load_test_callables(
            ctx.test_files, bare_name,
            project_root=ctx.project_root, func_key=func_key,
        )
        try:
            sr = run_function_sampling(
                node, func_key, cats, tests, lambda *_: None, budget_ms=budget_ms,
            )
            rd = sr.to_dict()
            rd['tests_loaded'] = len(tests)
            rd['is_pure'] = is_pure
            args_node = getattr(node, 'args', None)
            rd['parameter_count'] = len(args_node.args) if args_node else 0
            if diag:
            from dataclasses import asdict as _asdict
            if hasattr(diag, '__dataclass_fields__'):
                rd['discovery_diagnostics'] = _asdict(diag)
            else:
                rd['discovery_diagnostics'] = diag
            # Classify discovery state
            if not ctx.test_files:
                ds = 'NO_TEST_FILES'
            elif len(tests) == 0:
                ds = 'NO_TESTS_LINKED'
            elif sr.total_killed == 0 and sr.total_mutants == 0:
                ds = 'EQUIVALENT'
            elif sr.total_killed == 0 and sr.total_mutants > 0:
                ds = 'ZERO_KILLS'
            else:
                ds = 'OK'
            rd['discovery_state'] = ds
            save_cached_state(ctx.cache_dir, func_key, rd)
            result['profiled'] += 1
            result['killed'] += sr.total_killed
            result['survived'] += sr.total_survived
            result['discovery'][ds] = result['discovery'].get(ds, 0) + 1
        except Exception:
            result['errors'] += 1
    result['elapsed_s'] = round(time.monotonic() - start, 2)
    return result

project_root = os.path.abspath(PROJECT_DIR)
files = collect_python_files(project_root)
print(f'Found {len(files)} source files | Workers: {WORKERS} | Budget: {BUDGET_MS}ms')
cache_dir = Path(project_root) / '.lintgate' / 'mutation'
cache_dir.mkdir(parents=True, exist_ok=True)
work = [(project_root, f, BUDGET_MS, SKIP_CACHED) for f in files]
totals = {'profiled': 0, 'cached': 0, 'trivial': 0, 'killed': 0, 'survived': 0, 'errors': 0}
disc = {}
start = time.monotonic()
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(profile_single_file, w): w[1] for w in work}
    for i, future in enumerate(as_completed(futures), 1):
        rel = futures[future]
        try:
            r = future.result()
            for k in ('profiled', 'cached', 'trivial', 'killed', 'survived', 'errors'):
                totals[k] += r.get(k, 0)
            for ds, n in r.get('discovery', {}).items():
                disc[ds] = disc.get(ds, 0) + n
            print(f'[{i}/{len(files)}] {rel}: {r.get("profiled",0)} profiled, '
                  f'{r.get("cached",0)} cached, {r.get("elapsed_s","?")}s')
        except Exception as e:
            print(f'[{i}/{len(files)}] {rel}: FAILED - {e}')
            totals['errors'] += 1
elapsed = round(time.monotonic() - start, 1)
t = totals['killed'] + totals['survived']
kr = f'{totals["killed"]}/{t} ({totals["killed"]/t:.1%})' if t else 'N/A'
print(f'\n{"="*60}')
print(f'Done in {elapsed}s')
print(f'  Profiled:  {totals["profiled"]}')
print(f'  Cached:    {totals["cached"]}')
print(f'  Trivial:   {totals["trivial"]}')
print(f'  Errors:    {totals["errors"]}')
print(f'  Kill rate: {kr}')
print(f'  Discovery:')
for ds, n in sorted(disc.items()):
    print(f'    {ds}: {n}')
# Write summary JSON
import json as _json
summary = {'profiled': totals['profiled'], 'killed': totals['killed'],
           'survived': totals['survived'], 'kill_rate': round(totals['killed']/t, 4) if t else 0,
           'discovery_states': disc, 'errors': totals['errors'], 'elapsed_s': elapsed}
with open(cache_dir / 'sweep_summary.json', 'w') as sf:
    _json.dump(summary, sf, indent=2)
print(f'\nSummary written to .lintgate/mutation/sweep_summary.json')


## Step 3: Review & download


In [ ]:
import json, os
from pathlib import Path
from collections import Counter

cache_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
files = sorted(cache_dir.glob('*.json')) if cache_dir.exists() else []
files = [f for f in files if f.name != 'sweep_summary.json' and f.name != 'scheduler_state.json']
print(f'Total profiles: {len(files)}')

total_k, total_s = 0, 0
high = []
disc = Counter()
per_file_kills = Counter()
per_file_total = Counter()
for f in files:
    try:
        d = json.loads(f.read_text())
    except Exception:
        continue
    k, s = d.get('total_killed', 0), d.get('total_survived', 0)
    total_k += k
    total_s += s
    ds = d.get('discovery_state', 'UNKNOWN')
    disc[ds] += 1
    fk = d.get('function_key', '?')
    src = fk.split('::')[0] if '::' in fk else '?'
    per_file_kills[src] += k
    per_file_total[src] += k + s
    r = d.get('survival_rate', 0)
    if r > 0.5 and (k + s) > 0:
        high.append((fk, r))

t = total_k + total_s
print(f'Kill rate: {total_k}/{t} ({total_k/t:.1%})' if t else 'No mutants')
print(f'\nDiscovery states:')
for ds, n in disc.most_common():
    print(f'  {ds}: {n}')
print(f'\nHigh-survival functions ({len(high)}):')
for key, r in sorted(high, key=lambda x: -x[1])[:20]:
    print(f'  {r:.0%} {key}')
print(f'\nPer-file kill rates (worst 15):')
file_rates = [(f, per_file_kills[f], per_file_total[f]) for f in per_file_total if per_file_total[f] > 0]
file_rates.sort(key=lambda x: x[1]/x[2] if x[2] else 1)
for f, k, t in file_rates[:15]:
    print(f'  {k}/{t} ({k/t:.0%}) {f}')


## Step 4: Flat test coverage analysis (files > 400 LoC)


In [ ]:
import os, re
from pathlib import Path

SRC_DIRS = ['lintgate', 'mcp_tools']
TEST_DIR = os.path.join(PROJECT_DIR, 'tests')
MIN_LOC = 400

def count_lines(path):
    try:
        return sum(1 for _ in open(path, encoding='utf-8', errors='ignore'))
    except OSError:
        return 0

def find_test_file(base_name, test_dir):
    """Find a matching test file for a production module."""
    # Strip leading underscore for matching
    clean = base_name.lstrip('_')
    candidates = [
        f'test_{base_name}.py',
        f'test_{clean}.py',
    ]
    # Also check for partial matches
    for c in candidates:
        p = os.path.join(test_dir, c)
        if os.path.isfile(p):
            return p
    # Fuzzy: any test file containing the base name
    if os.path.isdir(test_dir):
        for f in os.listdir(test_dir):
            if f.startswith('test_') and clean in f and f.endswith('.py'):
                return os.path.join(test_dir, f)
    return None

# Collect all large production files
large_files = []
for subdir in SRC_DIRS:
    base = os.path.join(PROJECT_DIR, subdir)
    if not os.path.isdir(base):
        continue
    for dirpath, _, filenames in os.walk(base):
        for fn in sorted(filenames):
            if not fn.endswith('.py') or fn == '__init__.py':
                continue
            if fn.startswith('test_') or fn.endswith('_test.py'):
                continue
            full = os.path.join(dirpath, fn)
            loc = count_lines(full)
            if loc >= MIN_LOC:
                rel = os.path.relpath(full, PROJECT_DIR)
                large_files.append((rel, loc, fn))

# Match to test files and compute ratios
rows = []
for rel, src_loc, fn in large_files:
    base_name = fn[:-3]  # strip .py
    test_path = find_test_file(base_name, TEST_DIR)
    if test_path:
        test_loc = count_lines(test_path)
        test_name = os.path.basename(test_path)
    else:
        test_loc = 0
        test_name = 'NONE'
    ratio = test_loc / src_loc if src_loc > 0 else 0
    rows.append((rel, src_loc, test_loc, ratio, test_name))

# Sort by ratio ascending (worst coverage first)
rows.sort(key=lambda r: r[3])

# Print report
print(f'Test Coverage Analysis: {len(rows)} production files >= {MIN_LOC} LoC')
print(f'{"="*90}')
print(f'{"Source File":<55} {"Src":>5} {"Test":>5} {"Ratio":>6} {"Test File"}')
print(f'{"-"*90}')

no_test = [r for r in rows if r[4] == 'NONE']
low_cov = [r for r in rows if r[4] != 'NONE' and r[3] < 0.5]
good_cov = [r for r in rows if r[4] != 'NONE' and r[3] >= 0.5]

print(f'\n## NO TEST FILE ({len(no_test)} files, {sum(r[1] for r in no_test)} LoC)')
for rel, src, tst, ratio, tname in no_test:
    print(f'  {rel:<55} {src:>5}   ---    ---  {tname}')

print(f'\n## LOW COVERAGE <0.5x ({len(low_cov)} files)')
for rel, src, tst, ratio, tname in low_cov:
    print(f'  {rel:<55} {src:>5} {tst:>5} {ratio:>5.2f}x {tname}')

print(f'\n## ADEQUATE COVERAGE >=0.5x ({len(good_cov)} files)')
for rel, src, tst, ratio, tname in good_cov:
    print(f'  {rel:<55} {src:>5} {tst:>5} {ratio:>5.2f}x {tname}')

total_src = sum(r[1] for r in rows)
total_tst = sum(r[2] for r in rows)
print(f'\n{"="*90}')
print(f'Total: {total_src} src LoC, {total_tst} test LoC, {total_tst/total_src:.2f}x overall ratio')
print(f'No test file: {len(no_test)}/{len(rows)} ({len(no_test)/len(rows):.0%})')
print(f'Low coverage: {len(low_cov)}/{len(rows)} ({len(low_cov)/len(rows):.0%})')
print(f'Adequate:     {len(good_cov)}/{len(rows)} ({len(good_cov)/len(rows):.0%})')

# Write coverage summary JSON
import json
cov_summary = {
    'total_large_files': len(rows),
    'no_test_file': [{'file': r[0], 'loc': r[1]} for r in no_test],
    'low_coverage': [{'file': r[0], 'loc': r[1], 'test_loc': r[2], 'ratio': round(r[3], 2)} for r in low_cov],
    'adequate': [{'file': r[0], 'loc': r[1], 'test_loc': r[2], 'ratio': round(r[3], 2)} for r in good_cov],
    'totals': {'src_loc': total_src, 'test_loc': total_tst, 'ratio': round(total_tst/total_src, 3)},
}
cov_path = Path(PROJECT_DIR) / '.lintgate' / 'mutation' / 'coverage_analysis.json'
with open(cov_path, 'w') as cf:
    json.dump(cov_summary, cf, indent=2)
print(f'\nCoverage analysis written to .lintgate/mutation/coverage_analysis.json')


## Step 5: Download results


In [ ]:
import os
from pathlib import Path

mutation_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
mutation_dir.mkdir(parents=True, exist_ok=True)

zip_path = '/content/mutation_results.zip'
!cd {PROJECT_DIR} && zip -r {zip_path} .lintgate/mutation/ -x '*.DS_Store'

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'Size: {size_mb:.1f} MB')
    try:
        from google.colab import files
        files.download(zip_path)
        print('Download started!')
    except ImportError:
        print(f'Not in Colab. Results at: {zip_path}')
else:
    print('No zip created. Check that the sweep produced results.')

print(f'\nTo apply: cd . && unzip ~/Downloads/mutation_results.zip')
